# 04 — silver_order_items
**Entrée :** `bronze.order_items` — **Sortie :** `silver.silver_order_items` (112 650 lignes)
**Typage :** prix et frais de port en `DECIMAL(10,2)` (jamais de DOUBLE pour de l'argent), date limite en TIMESTAMP.
**Indicateurs :** `flag_freight_gt_price` (4 124), `flag_freight_zero` (383).
**Clé :** (`order_id`, `order_item_id`).

In [0]:
CATALOG = "workspace"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
# Les tables Bronze sont supposées s'appeler : orders, customers, order_items, order_payments,
# order_reviews, products, sellers, product_category_name_translation.
# Si les tiennes ont d'autres noms, change-les dans les requêtes (Ctrl+F).

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")


def scalar(sql):
    """Exécute une requête qui renvoie une seule valeur."""
    return spark.sql(sql).collect()[0][0]


def check(label, obtenu, attendu):
    """Compare ce qu'on obtient à ce qu'on attend (valeurs de ton document de travail)."""
    ok = obtenu == attendu
    print(("✅ OK     " if ok else "❌ ÉCART  ") + f"{label} : obtenu = {obtenu} | attendu = {attendu}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {SILVER}.silver_order_items AS
WITH i AS (
    SELECT
        order_id,
        CAST(TRY_CAST(order_item_id AS DOUBLE) AS INT) AS order_item_id,
        product_id,
        seller_id,
        TRY_CAST(NULLIF(TRIM(shipping_limit_date), '') AS TIMESTAMP) AS shipping_limit_ts,
        TRY_CAST(price AS DECIMAL(10,2))         AS price,
        TRY_CAST(freight_value AS DECIMAL(10,2)) AS freight_value,
        _loaded_at
    FROM {BRONZE}.order_items
)
SELECT
    i.*,
    COALESCE(freight_value > price, FALSE) AS flag_freight_gt_price,
    COALESCE(freight_value = 0, FALSE)     AS flag_freight_zero,
    _loaded_at AS _bronze_loaded_at,
    current_timestamp() AS _silver_loaded_at
FROM i
""")

In [0]:
t = f"{SILVER}.silver_order_items"
check("lignes", scalar(f"SELECT COUNT(*) FROM {t}"), 112650)
check("commandes distinctes", scalar(f"SELECT COUNT(DISTINCT order_id) FROM {t}"), 98666)
check("clé (order_id, order_item_id) en double", scalar(
    f"SELECT COUNT(*) - COUNT(DISTINCT order_id, order_item_id) FROM {t}"), 0)
check("prix NULL ou <= 0", scalar(f"SELECT COUNT(*) FROM {t} WHERE price IS NULL OR price <= 0"), 0)
check("frais de port NULL", scalar(f"SELECT COUNT(*) FROM {t} WHERE freight_value IS NULL"), 0)
check("flag_freight_gt_price", scalar(f"SELECT COUNT(*) FROM {t} WHERE flag_freight_gt_price"), 4124)
check("flag_freight_zero", scalar(f"SELECT COUNT(*) FROM {t} WHERE flag_freight_zero"), 383)

# La somme des prix doit être égale à Bronze (c'est le test de réconciliation)
s_bronze = scalar(f"SELECT ROUND(SUM(TRY_CAST(price AS DOUBLE)), 2) FROM {BRONZE}.order_items")
s_silver = float(scalar(f"SELECT SUM(price) FROM {t}"))
check("somme des prix Bronze = Silver", round(s_silver, 2), round(s_bronze, 2))
print("Rappel : le chiffre d'affaires attendu (prix seuls) est d'environ 13 591 640.")

## Pour ton journal (à rédiger avec tes mots)
1. Pourquoi `TRY_CAST` plutôt que `CAST` ?
2. Pourquoi on garde les lignes anormales avec un indicateur `flag_…` ?
3. Pourquoi Silver a le même nombre de lignes que Bronze ?